In [ ]:
import os

DATA_ROOT = os.environ.get("DATA_ROOT", "./data/wildlife_dataset")

TRAIN_DIR = os.path.join(DATA_ROOT, "datawildlife_train")
VAL_DIR   = os.path.join(DATA_ROOT, "datawildlife_val")
TEST_DIR  = os.path.join(DATA_ROOT, "datawildlife_test")

# create wildlife_detect.yaml from wildlife.yaml
YAML_DIR = os.path.join(DATA_ROOT, "..")
BASE_YAML = os.path.join(YAML_DIR, "wildlife.yaml")
DETECT_YAML = os.path.join(YAML_DIR, "wildlife_detect.yaml")

def to_yaml_path(p):
    return p.replace("\\", "/")

if os.path.exists(DETECT_YAML):
    print(f"Already exists, skip: {DETECT_YAML}")
else:
    with open(BASE_YAML, 'r') as f:
        base_content = f.read()

    detect_yaml = (
        f"train: {to_yaml_path(TRAIN_DIR)}\n"
        f"val: {to_yaml_path(VAL_DIR)}\n\n"
        f"{base_content}"
    )

    with open(DETECT_YAML, 'w') as f:
        f.write(detect_yaml)

    print(f"Created: {DETECT_YAML}")

In [ ]:
import shutil
import os

datasets = [
    'motiontrack',
    'wildlife-trainval'
]

base_input = '/kaggle/input/datasets/quthiu'
base_output = '/kaggle/working'

for name in datasets:
    src = f'{base_input}/{name}'
    dst = f'{base_output}/{name}'
    print(f'Copying {name}...')
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f'Done: {dst}')

In [ ]:
!pip install -r /kaggle/working/motiontrack/MotionTrack/requirements.txt

In [ ]:
yaml_content = """
train: /kaggle/working/wildlife-trainval/datawildlife_test/datawildlife_test
val: /kaggle/working/wildlife-trainval/datawildlife_val/datawildlife_val
nc: 3
names: ['elephant', 'zebra', 'giraffe']
"""

with open('/kaggle/working/wildlife.yaml', 'w') as f:
    f.write(yaml_content)

print("Created: /kaggle/working/wildlife.yaml")

In [ ]:
with open('/kaggle/working/motiontrack/MotionTrack/tools/train.py', 'r') as f:
    content = f.read()

content = content.replace(
    'torch.load(weights).get',
    'torch.load(weights, weights_only=False).get'
).replace(
    'torch.load(weights, map_location=device)',
    'torch.load(weights, map_location=device, weights_only=False)'
)

with open('/kaggle/working/motiontrack/MotionTrack/tools/train.py', 'w') as f:
    f.write(content)

In [ ]:
with open('/kaggle/working/motiontrack/MotionTrack/utils/loss.py', 'r') as f:
    content = f.read()

content = content.replace(
    'fg_mask_inboxes = matching_matrix.sum(0) > 0.0',
    'fg_mask_inboxes = (matching_matrix.sum(0) > 0.0).to(from_which_layer.device)'
)

with open('/kaggle/working/motiontrack/MotionTrack/utils/loss.py', 'w') as f:
    f.write(content)

In [ ]:
with open('/kaggle/working/motiontrack/MotionTrack/utils/metrics.py', 'r') as f:
    content = f.read()

content = content.replace('np.trapz', 'np.trapezoid')

with open('/kaggle/working/motiontrack/MotionTrack/utils/metrics.py', 'w') as f:
    f.write(content)

In [ ]:
with open('/kaggle/working/motiontrack/MotionTrack/data/hyp.scratch.tiny.yaml', 'r') as f:
    content = f.read()

content = content.replace('lr0: 0.01', 'lr0: 0.005')
content = content.replace('warmup_epochs: 3.0', 'warmup_epochs: 2.0')

with open('/kaggle/working/motiontrack/MotionTrack/data/hyp.scratch.tiny.yaml', 'w') as f:
    f.write(content)

In [ ]:
with open('/kaggle/working/motiontrack/MotionTrack/tools/train.py', 'r') as f:
    content = f.read()
    
content = content.replace(
    "epochs += ckpt['epoch']  # finetune additional epochs",
    "epochs = opt.epochs  # use specified epochs for fine-tuning"
)

with open('/kaggle/working/motiontrack/MotionTrack/tools/train.py', 'w') as f:
    f.write(content)

In [ ]:
#resume
%cd /kaggle/working/motiontrack/MotionTrack
import os
os.environ['WANDB_DISABLED'] = 'true'
os.environ['WANDB_MODE'] = 'disabled'

!python tools/train.py --resume /kaggle/input/datasets/quthiu/for-resume/stop_here/weights/last.pt